# 04 · Subset, k-core filtering and re-indexing

### What this notebook does
1. Applies the subset chosen in notebook 02 (`interim/subset_choice.json`) to the **cleaned** interactions.
2. Re-applies **iterative 5-core** filtering until stable, logging every iteration.
3. **Re-indexes** IDs: items → `1 … N` (0 is reserved for padding), users → `0 … U-1`. Saves both mappings.
4. Produces the **before / after statistics table** for the report (raw → cleaned → subset → final).

### Why
- Cleaning (notebook 03) and subsetting both remove interactions, which can push users or items below 5.
  One pass is not enough: removing a sparse item can make a user sparse, so we iterate until nothing changes.
- Embedding tables need consecutive integer IDs. Padding index 0 lets SASRec left-pad short sequences.
- The report's Stage 1 output is "Reproducible Model 0 **and data statistics**": the table below is that statistic.

**Inputs:** `interim/ratings_clean.parquet`, `interim/meta_clean.parquet`, `interim/subset_choice.json`
**Outputs:** `interim/interactions_kcore.parquet`, `processed/item_map.parquet`, `processed/user_map.parquet`,
`processed/items.parquet`, `reports/dataset_statistics.csv`

In [ ]:
# ---------------- Shared setup (identical in every notebook) ----------------
# All notebooks read and write inside one project folder (BASE).
# Colab: BASE lives in your Google Drive so outputs survive runtime restarts.
# Local Jupyter: BASE is ./298A_recsys next to the notebooks.
# Set the environment variable RECSYS_BASE to use any other folder.
import os, sys, json, re, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules
if os.environ.get('RECSYS_BASE'):
    BASE = Path(os.environ['RECSYS_BASE'])
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/298A_recsys')
else:
    BASE = Path('./298A_recsys').resolve()

RAW, INTERIM, PROCESSED = BASE / 'raw', BASE / 'interim', BASE / 'processed'
REPORTS, FIGS = BASE / 'reports', BASE / 'reports' / 'figures'
# Large downloads go to fast local disk in Colab (not Drive); re-downloaded if the runtime resets.
DOWNLOADS = Path(os.environ.get('RECSYS_DOWNLOAD_DIR', '/content/downloads' if IN_COLAB else str(BASE / 'downloads')))
for d in [RAW, INTERIM, PROCESSED, REPORTS, FIGS, DOWNLOADS]:
    d.mkdir(parents=True, exist_ok=True)

# Plot style: one hue per single-series chart, light recessive grid, no chart junk.
PRIMARY, ACCENT, MUTED = '#2a78d6', '#eb6834', '#8a8984'
plt.rcParams.update({
    'figure.figsize': (8, 4), 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'axes.prop_cycle': matplotlib.cycler(color=[PRIMARY, ACCENT, '#1baf7a', '#eda100']),
})

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGS / f'{name}.png', bbox_inches='tight')
    plt.show()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_config():
    cfg_path = BASE / 'config.json'
    if not cfg_path.exists():
        raise FileNotFoundError('config.json not found - run 01_data_acquisition.ipynb first.')
    return json.loads(cfg_path.read_text())

print('Project folder:', BASE)

In [ ]:
CFG = load_config()
SEED, K = CFG['seed'], CFG['kcore']
ratings = pd.read_parquet(INTERIM / 'ratings_clean.parquet')
meta = pd.read_parquet(INTERIM / 'meta_clean.parquet')
raw = pd.read_parquet(RAW / 'ratings_5core.parquet', columns=['user_id', 'parent_asin', 'rating', 'timestamp'])
choice = json.loads((INTERIM / 'subset_choice.json').read_text())
print('Subset choice:', choice['kind'], '=', choice['value'])
print('Reason:', choice['reason'])

## 1. Apply the subset
- **category**: keep interactions whose item has `cat_l1 == value`.
- **user_sample**: draw the same random users as in notebook 02 (same seed, same sorted user list), from the
  original 5-core users so the sample matches the EDA exactly.

In [ ]:
if choice['kind'] == 'category':
    items_in_cat = set(meta.loc[meta.cat_l1 == choice['value'], 'parent_asin'])
    subset = ratings[ratings.parent_asin.isin(items_in_cat)]
elif choice['kind'] == 'user_sample':
    all_users = np.sort(raw.user_id.unique().astype(str))
    rng = np.random.default_rng(choice['seed'])
    chosen = rng.choice(all_users, size=min(int(choice['value']), len(all_users)), replace=False)
    subset = ratings[ratings.user_id.isin(set(chosen))]
else:
    raise ValueError(f'Unknown subset kind {choice["kind"]}')
print(f'Subset before k-core: {len(subset):,} interactions, {subset.user_id.nunique():,} users, {subset.parent_asin.nunique():,} items')

## 2. Iterative k-core

In [ ]:
def kcore(df, k, user_col='user_id', item_col='parent_asin'):
    history = []
    while True:
        n = len(df)
        uc = df.groupby(user_col, observed=True)[user_col].transform('size')
        ic = df.groupby(item_col, observed=True)[item_col].transform('size')
        df = df[(uc >= k) & (ic >= k)]
        history.append({'iteration': len(history) + 1, 'rows_before': n, 'rows_after': len(df),
                        'users': df[user_col].nunique(), 'items': df[item_col].nunique()})
        if len(df) == n:
            return df, pd.DataFrame(history)

final, kcore_log = kcore(subset, K)
display(kcore_log)
assert final.groupby('user_id').size().min() >= K and final.groupby('parent_asin').size().min() >= K
print(f'Final: {len(final):,} interactions | {final.user_id.nunique():,} users | {final.parent_asin.nunique():,} items')

exp = choice.get('expected_after_kcore', {})
if exp:
    print('Compared with the EDA estimate (before cleaning):',
          {k: f'{v:,} expected' for k, v in exp.items()})

## 3. Re-index IDs
Items are numbered in sorted `parent_asin` order and users in sorted `user_id` order, so the mapping is
deterministic: re-running the notebook always gives the same integers.

In [ ]:
item_ids = np.sort(final.parent_asin.unique())
user_ids = np.sort(final.user_id.unique())
item_map = pd.DataFrame({'item_idx': np.arange(1, len(item_ids) + 1, dtype=np.int32), 'parent_asin': item_ids})
user_map = pd.DataFrame({'user_idx': np.arange(len(user_ids), dtype=np.int32), 'user_id': user_ids})

final = (final.merge(item_map, on='parent_asin', how='left')
              .merge(user_map, on='user_id', how='left'))
assert final.item_idx.notna().all() and final.user_idx.notna().all()
assert final.item_idx.min() == 1 and final.item_idx.max() == len(item_ids), 'item ids must be 1..N (0 = padding)'
assert final.user_idx.min() == 0 and final.user_idx.max() == len(user_ids) - 1
N_ITEMS, N_USERS = len(item_ids), len(user_ids)
print(f'N_ITEMS = {N_ITEMS:,} (embedding table size = N_ITEMS + 1 for padding) | N_USERS = {N_USERS:,}')

## 4. Item table for the final catalog
One row per `item_idx` with the cleaned metadata. Notebook 06 builds the semantic text from this table.

In [ ]:
items = item_map.merge(meta, on='parent_asin', how='left')
assert len(items) == N_ITEMS and items.title.notna().all()
items['n_interactions'] = items.item_idx.map(final.groupby('item_idx').size()).astype(int)
print('Level-2 categories in the final catalog:', items.cat_l2.nunique())
items[['item_idx', 'parent_asin', 'title', 'brand', 'cat_l1', 'cat_l2', 'n_interactions']].head()

## 5. Dataset statistics table (for the report)

In [ ]:
def describe(df, name, user_col='user_id', item_col='parent_asin'):
    pu = df.groupby(user_col).size()
    pi = df.groupby(item_col).size()
    ts = pd.to_datetime(df.timestamp, unit='ms', utc=True)
    u, i, n = df[user_col].nunique(), df[item_col].nunique(), len(df)
    return {'stage': name, 'users': u, 'items': i, 'interactions': n,
            'density_%': round(100 * n / (u * i), 4),
            'avg_seq_len': round(pu.mean(), 2), 'median_seq_len': float(pu.median()), 'max_seq_len': int(pu.max()),
            'avg_item_pop': round(pi.mean(), 2), 'median_item_pop': float(pi.median()),
            'mean_rating': round(df.rating.mean(), 3),
            'start': ts.min().date().isoformat(), 'end': ts.max().date().isoformat()}

stats = pd.DataFrame([
    describe(raw, 'raw 5-core (official)'),
    describe(ratings, 'after cleaning'),
    describe(subset, f'subset: {choice["value"]}'),
    describe(final, f'final ({K}-core, re-indexed)'),
]).set_index('stage')
stats.to_csv(REPORTS / 'dataset_statistics.csv')
stats.T

In [ ]:
pu = final.groupby('user_idx').size()
pi = final.groupby('item_idx').size()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(pu.clip(upper=pu.quantile(0.99)), bins=40, color=PRIMARY)
axes[0].set_title('Final: interactions per user'); axes[0].set_xlabel('interactions (clipped at p99)')
axes[1].hist(pi.clip(upper=pi.quantile(0.99)), bins=40, color=PRIMARY)
axes[1].set_title('Final: interactions per item'); axes[1].set_xlabel('interactions (clipped at p99)')
savefig('04_final_distributions')

## Save

In [ ]:
cols = ['user_idx', 'item_idx', 'rating', 'timestamp', 'row_id', 'user_id', 'parent_asin']
final[cols].sort_values(['user_idx', 'timestamp', 'row_id']).reset_index(drop=True).to_parquet(INTERIM / 'interactions_kcore.parquet', index=False)
item_map.to_parquet(PROCESSED / 'item_map.parquet', index=False)
user_map.to_parquet(PROCESSED / 'user_map.parquet', index=False)
items.to_parquet(PROCESSED / 'items.parquet', index=False)
kcore_log.to_csv(REPORTS / 'kcore_iterations.csv', index=False)
save_json({'n_items': N_ITEMS, 'n_users': N_USERS, 'n_interactions': len(final), 'padding_idx': 0,
           'subset': choice}, INTERIM / 'index_info.json')
print('Saved interactions, ID maps, item table and statistics.')

**Next:** `05_split_eval_groups_clients.ipynb`